# PyHealth Tutorial 07: Clinical text: classification and medical coding

**What you will learn**
- How to fine-tune a Hugging Face language model on clinical notes with PyHealth
- Multiclass text classification: which specialty wrote this report?
- Multilabel medical coding: which ICD codes does this note support?
- Why a simple TF-IDF baseline belongs next to every language model

**Time:** about 25 minutes on a free Colab CPU runtime.  
**Before this:** Tutorials 00 and 02. A GPU runtime (Runtime > Change runtime type > T4 GPU) makes training much faster.

All data used here is synthetic or public, so every cell runs without credentials.

## Setup

These tutorials use features from the upcoming PyHealth 2.1 release, so for now we install the latest version straight from GitHub.

**On Colab this takes two clicks of Runtime > Run all.** The first run installs PyHealth and restarts the runtime, because Colab has already loaded older versions of some packages (such as numpy). Colab then says "Your session crashed": that is expected. Choose Run all again and the tutorial runs through. On your own machine, install PyHealth once (`pip install "pyhealth[xgboost]"`) and this cell does nothing.

In [ ]:
# First run: installs PyHealth, then restarts the runtime once so Python
# loads the new package versions. Then choose Runtime > Run all again;
# this cell will see PyHealth is installed and skip the install.
import importlib.util
import os
import subprocess
import sys

PYHEALTH = "pyhealth[xgboost] @ git+https://github.com/sunlabuiuc/PyHealth.git"

if importlib.util.find_spec("pyhealth") is None:
    print("Installing PyHealth (1-2 minutes)...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", PYHEALTH],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-5000:], result.stderr[-5000:])
        raise RuntimeError("Installing PyHealth failed; see the log above.")
    print("Installed. Restarting the runtime now; then choose Runtime > Run all again.")
    os.kill(os.getpid(), 9)  # Colab restarts the runtime automatically
else:
    import logging
    import pyhealth
    logging.getLogger("pyhealth").propagate = False  # print each log line once
    print("PyHealth is installed and ready.")

## Text is just another field

For PyHealth, a note is one more input field. The `text` processor
keeps it as a string, and `TransformersModel` wraps any Hugging Face
encoder: it tokenizes the text (first 256 tokens), encodes it, and
adds a prediction head that matches the task's label (multiclass,
multilabel, binary).

Model size matters on a free runtime. We pick automatically: a
clinical BERT (110M parameters, trained on MIMIC notes) on a GPU, or a
small general BERT (11M parameters) on a CPU.

In [ ]:
import torch

if torch.cuda.is_available():
    MODEL_NAME, EPOCHS = "emilyalsentzer/Bio_ClinicalBERT", 3
else:
    MODEL_NAME, EPOCHS = "google/bert_uncased_L-4_H-256_A-4", 2
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| model:", MODEL_NAME)

## Part A. Which specialty wrote this report?

The [MTSamples](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions)
collection has about 5,000 transcribed medical reports, each labelled
with a medical specialty. These are sample documents, not patient
records, so each report is its own "patient".

In [ ]:
import urllib.request
import zipfile

urllib.request.urlretrieve(
    "https://storage.googleapis.com/pyhealth/medical_transcriptions_data/MedicalTranscriptions.zip",
    "MedicalTranscriptions.zip",
)
zipfile.ZipFile("MedicalTranscriptions.zip").extractall(".")

from pyhealth.datasets import MedicalTranscriptionsDataset

mts = MedicalTranscriptionsDataset(root="MedicalTranscriptions", cache_dir="pyhealth_cache")
mts.stats()

The dataset comes with a default task, text in and specialty out:

In [ ]:
print(mts.default_task.input_schema, "->", mts.default_task.output_schema)
reports = mts.set_task()

sample = reports[0]
print(f"{len(reports)} reports")
print("text:", sample["transcription"][:300], "...")
labels = reports.output_processors["medical_specialty"].label_vocab
print(f"\n{len(labels)} specialties, e.g. {list(labels)[:6]}")

Each report is its own patient, so a sample-level split is fine here.
Then the usual PyHealth steps: model, trainer, evaluation. For
multiclass labels the trainer reports accuracy and F1; macro F1
weights every specialty equally, so rare specialties count as much as
common ones.

Loading the model prints a table of "unexpected" weights. That is
normal: the pretraining heads of the checkpoint are dropped, and
PyHealth adds a new classification head for our labels.

In [ ]:
from pyhealth.datasets import get_dataloader, split_by_sample
from pyhealth.models import TransformersModel
from pyhealth.trainer import Trainer

train_ds, val_ds, test_ds = split_by_sample(reports, [0.7, 0.1, 0.2], seed=0)
model = TransformersModel(dataset=reports, model_name=MODEL_NAME)
trainer = Trainer(model=model, metrics=["accuracy", "f1_macro", "f1_weighted"])
trainer.train(
    train_dataloader=get_dataloader(train_ds, batch_size=16, shuffle=True),
    val_dataloader=get_dataloader(val_ds, batch_size=32),
    epochs=EPOCHS,
    monitor="f1_weighted",
    optimizer_params={"lr": 5e-5},
)
bert_scores = trainer.evaluate(get_dataloader(test_ds, batch_size=32))
print(bert_scores)

### A baseline to beat

Bag-of-words models are fast and often strong on text classification.
A TF-IDF + logistic regression baseline on the same split takes
seconds; a language model should clearly beat it to justify its cost.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def texts_and_labels(ds):
    return [ds[i]["transcription"] for i in range(len(ds))], [int(ds[i]["medical_specialty"]) for i in range(len(ds))]

x_train, y_train = texts_and_labels(train_ds)
x_test, y_test = texts_and_labels(test_ds)
tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)
clf = LogisticRegression(max_iter=2000).fit(tfidf.fit_transform(x_train), y_train)
pred = clf.predict(tfidf.transform(x_test))
print(f"TF-IDF + LR   accuracy {accuracy_score(y_test, pred):.3f}  "
      f"F1 macro {f1_score(y_test, pred, average='macro'):.3f}  "
      f"F1 weighted {f1_score(y_test, pred, average='weighted'):.3f}")
print(f"{MODEL_NAME.split('/')[-1]:13s} accuracy {bert_scores['accuracy']:.3f}  "
      f"F1 macro {bert_scores['f1_macro']:.3f}  F1 weighted {bert_scores['f1_weighted']:.3f}")

Scores on this dataset are modest for any model: several "specialties"
(Surgery, Consult, Discharge Summary) describe the *kind* of document
rather than the field, and overlap heavily with the others. Looking at
the labels before modelling saves a lot of tuning.

## Part B. Medical coding from notes

Medical coding assigns ICD codes to a hospital stay from its
documentation. It is a **multilabel** problem: each note supports
several codes at once. PyHealth's `MIMIC3ICD9Coding` task joins a
patient's notes into one text and collects the diagnosis and
procedure codes as labels:

In [ ]:
from pyhealth.datasets import MIMIC3Dataset
from pyhealth.tasks import MIMIC3ICD9Coding

mimic = MIMIC3Dataset(
    root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
    tables=["diagnoses_icd", "procedures_icd", "noteevents"],
    cache_dir="pyhealth_cache",
)
coding_task = MIMIC3ICD9Coding()
print(coding_task.input_schema, "->", coding_task.output_schema)
notes = mimic.set_task(coding_task)
codes = notes.output_processors["icd_codes"].label_vocab
print(f"{len(notes)} samples, {len(codes)} distinct codes")
print("first sample has", int(notes[0]["icd_codes"].sum()), "codes")

With thousands of possible codes, most appearing only a few times,
coding is hard. Two habits from practice:

- Report **micro F1** (dominated by frequent codes) and **macro F1**
  (every code equal), plus ranking metrics such as precision@k.
- Restrict to the most frequent codes (for example the top 50) for a
  first model, as most published MIMIC coding benchmarks do.

The training code is identical to Part A; only the task changes. (On
synthetic notes the text carries little information about the codes,
so expect low scores; the workflow is what transfers to real MIMIC.)

In [ ]:
from pyhealth.datasets import split_by_patient

train_n, val_n, test_n = split_by_patient(notes, [0.7, 0.1, 0.2], seed=0)
coder = TransformersModel(dataset=notes, model_name=MODEL_NAME)
coding_trainer = Trainer(model=coder, metrics=["f1_micro", "f1_macro", "pr_auc_samples"])
coding_trainer.train(
    train_dataloader=get_dataloader(train_n, batch_size=16, shuffle=True),
    val_dataloader=get_dataloader(val_n, batch_size=32),
    epochs=1,
    monitor="pr_auc_samples",
    optimizer_params={"lr": 5e-5},
)
print(coding_trainer.evaluate(get_dataloader(test_n, batch_size=32)))

## Summary

- A `text` field plus `TransformersModel(model_name=...)` fine-tunes
  any Hugging Face encoder; the task decides multiclass vs multilabel.
- Pick the model size for your hardware; use a GPU runtime for
  full-size clinical models.
- Always compare with a TF-IDF baseline, and inspect the labels before
  tuning.
- Medical coding is multilabel and long-tailed: report micro and macro
  metrics, and start with frequent codes.

## Where to go next

**Next:** Tutorial 08, *Contributing a dataset or task to PyHealth*.

The whole series:

00. Quickstart: your first clinical prediction model
01. Datasets: patients, events and your own data
02. Tasks and processors: from patients to model-ready samples
03. Training and evaluating models properly
04. Choosing a model: from logistic regression to XGBoost
05. Interpreting predictions
06. Medical codes: lookups, mappings and groupers
07. Clinical text: classification and medical coding
08. Contributing a dataset or task to PyHealth

Questions or problems? Open an issue at https://github.com/sunlabuiuc/PyHealth/issues. If PyHealth is useful to you, a star on GitHub helps the project.